In [24]:
# CELL 1 - READ BRONZE TABLE

bronze_prescriptions_df  = spark.table("bronze_prescriptions")

print("Bronze Rows:", bronze_prescriptions_df.count())
bronze_prescriptions_df.printSchema()
print(bronze_prescriptions_df.columns)

display(bronze_prescriptions_df.limit(20))

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 26, Finished, Available, Finished, False)

Bronze Rows: 486634
root
 |-- prescription_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- prescribing_provider_id: string (nullable = true)
 |-- pharmacy_id: string (nullable = true)
 |-- prescriptiontimestamp: string (nullable = true)
 |-- drugndccode: string (nullable = true)
 |-- drug_name: string (nullable = true)
 |-- dosage: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- refills_authorized: string (nullable = true)
 |-- dispense_s: string (nullable = true)
 |-- denial_reason: string (nullable = true)
 |-- pharmacy_ip_address: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

['prescription_id', 'patient_id', 'prescribing_provider_id', 'pharmacy_id', 'prescriptiontimestamp', 'drugndccode', 'drug_name', 'dosage', 'quantity', 'refills_authorized', 'dispense_s', 

SynapseWidget(Synapse.DataFrame, e556a535-35ef-4657-a55a-383cdb7a035c)

In [25]:
# CELL 2 - STANDARDIZE COLUMN NAMES AND TRIM STRING VALUES

from pyspark.sql.functions import col, trim

prescriptions_standardized_df = (
    bronze_prescriptions_df
    .withColumnRenamed(
        "prescriptiontimestamp",
        "prescription_timestamp"
    )
    .withColumnRenamed(
        "drugndccode",
        "drug_ndc_code"
    )
    .withColumnRenamed(
        "dispense_s",
        "dispense_status"
    )
)

# Remove leading and trailing spaces from all string columns
for field in prescriptions_standardized_df.schema.fields:
    if field.dataType.simpleString() == "string":
        prescriptions_standardized_df = (
            prescriptions_standardized_df
            .withColumn(field.name, trim(col(field.name)))
        )

print("Standardized Rows:", prescriptions_standardized_df.count())
print(prescriptions_standardized_df.columns)

display(prescriptions_standardized_df.limit(20))

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 27, Finished, Available, Finished, False)

Standardized Rows: 486634
['prescription_id', 'patient_id', 'prescribing_provider_id', 'pharmacy_id', 'prescription_timestamp', 'drug_ndc_code', 'drug_name', 'dosage', 'quantity', 'refills_authorized', 'dispense_status', 'denial_reason', 'pharmacy_ip_address', 'unexpected_source_field', 'batch_id', 'source_file_name', 'ingestion_timestamp']


SynapseWidget(Synapse.DataFrame, ea4a27ca-e861-4939-91a4-4ffc1a37f25b)

In [26]:
# CELL 3 - CLEAN VALUES AND CONVERT DATA TYPES
# Preserves raw source values for validation and quarantine

from pyspark.sql.functions import (
    col,
    when,
    upper,
    lower,
    trim,
    regexp_replace,
    to_timestamp
)

# Values that should be treated as missing
null_like_values = [
    "",
    "null",
    "none",
    "na",
    "n/a",
    "no value",
    "undefined",
    "unknown"
]

prescriptions_cleaned_df = prescriptions_standardized_df

# Standardize all string columns:
# 1. Trim outer spaces
# 2. Replace repeated internal spaces with one space
# 3. Convert known null-like values to actual NULL
for field in prescriptions_cleaned_df.schema.fields:

    if field.dataType.simpleString() == "string":

        cleaned_value = regexp_replace(
            trim(col(field.name)),
            r"\s+",
            " "
        )

        prescriptions_cleaned_df = (
            prescriptions_cleaned_df
            .withColumn(
                field.name,
                when(
                    lower(cleaned_value).isin(null_like_values),
                    None
                ).otherwise(cleaned_value)
            )
        )

# Preserve original values before datatype conversion.
# Cell 4 will use these columns to distinguish missing
# values from malformed source values.
prescriptions_cleaned_df = (
    prescriptions_cleaned_df

    .withColumn(
        "prescription_timestamp_raw",
        col("prescription_timestamp")
    )
    .withColumn(
        "quantity_raw",
        col("quantity")
    )
    .withColumn(
        "refills_authorized_raw",
        col("refills_authorized")
    )

    # Standardize identifier casing
    .withColumn(
        "prescription_id",
        upper(col("prescription_id"))
    )
    .withColumn(
        "patient_id",
        upper(col("patient_id"))
    )
    .withColumn(
        "prescribing_provider_id",
        upper(col("prescribing_provider_id"))
    )
    .withColumn(
        "pharmacy_id",
        upper(col("pharmacy_id"))
    )
    .withColumn(
        "drug_ndc_code",
        upper(col("drug_ndc_code"))
    )

    # Standardize status casing
    .withColumn(
        "dispense_status",
        upper(col("dispense_status"))
    )

    # Convert valid timestamp strings
    .withColumn(
        "prescription_timestamp",
        to_timestamp(col("prescription_timestamp"))
    )

    # Cast numeric values only when they contain whole numbers
    .withColumn(
        "quantity",
        when(
            col("quantity_raw").rlike(r"^[+-]?\d+$"),
            col("quantity_raw").cast("long")
        ).otherwise(None)
    )
    .withColumn(
        "refills_authorized",
        when(
            col("refills_authorized_raw").rlike(r"^[+-]?\d+$"),
            col("refills_authorized_raw").cast("int")
        ).otherwise(None)
    )
)

print("Cleaned Rows:", prescriptions_cleaned_df.count())

prescriptions_cleaned_df.printSchema()

display(
    prescriptions_cleaned_df.select(
        "prescription_id",
        "patient_id",
        "prescribing_provider_id",
        "pharmacy_id",
        "prescription_timestamp_raw",
        "prescription_timestamp",
        "drug_ndc_code",
        "drug_name",
        "quantity_raw",
        "quantity",
        "refills_authorized_raw",
        "refills_authorized",
        "dispense_status"
    ).limit(20)
)

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 28, Finished, Available, Finished, False)

Cleaned Rows: 486634
root
 |-- prescription_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- prescribing_provider_id: string (nullable = true)
 |-- pharmacy_id: string (nullable = true)
 |-- prescription_timestamp: timestamp (nullable = true)
 |-- drug_ndc_code: string (nullable = true)
 |-- drug_name: string (nullable = true)
 |-- dosage: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- refills_authorized: integer (nullable = true)
 |-- dispense_status: string (nullable = true)
 |-- denial_reason: string (nullable = true)
 |-- pharmacy_ip_address: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- prescription_timestamp_raw: string (nullable = true)
 |-- quantity_raw: string (nullable = true)
 |-- refills_authorized_raw: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 02bf2307-f352-44e7-9457-49e3cb7b3d10)

In [27]:
# CELL 4 - APPLY DATA QUALITY VALIDATION RULES

from pyspark.sql.functions import (
    col,
    when,
    lit,
    concat_ws,
    lower,
    to_date,
    current_date
)

# Accepted prescription statuses
allowed_dispense_statuses = [
    "FILLED",
    "PENDING",
    "DISPENSED",
    "DENIED"
]

prescriptions_validated_df = (
    prescriptions_cleaned_df
    .withColumn(
        "error_reason",
        concat_ws(
            "; ",

            # Required identifiers
            when(
                col("prescription_id").isNull(),
                lit("Missing prescription_id")
            ),
            when(
                col("prescription_id").isNotNull()
                & ~col("prescription_id").rlike(r"^RX_[0-9]+$"),
                lit("Invalid prescription_id format")
            ),

            when(
                col("patient_id").isNull(),
                lit("Missing patient_id")
            ),
            when(
                col("patient_id").isNotNull()
                & ~col("patient_id").rlike(r"^PAT_[0-9]+$"),
                lit("Invalid patient_id format")
            ),

            when(
                col("prescribing_provider_id").isNull(),
                lit("Missing prescribing_provider_id")
            ),
            when(
                col("prescribing_provider_id").isNotNull()
                & ~col("prescribing_provider_id")
                    .rlike(r"^PROV_[0-9]+$"),
                lit("Invalid prescribing_provider_id format")
            ),

            when(
                col("pharmacy_id").isNull(),
                lit("Missing pharmacy_id")
            ),
            when(
                col("pharmacy_id").isNotNull()
                & ~col("pharmacy_id").rlike(r"^PHARM_[0-9]+$"),
                lit("Invalid pharmacy_id format")
            ),

            # Timestamp validation
            when(
                col("prescription_timestamp_raw").isNull(),
                lit("Missing prescription_timestamp")
            ),
            when(
                col("prescription_timestamp_raw").isNotNull()
                & col("prescription_timestamp").isNull(),
                lit("Invalid prescription_timestamp format")
            ),
            when(
                col("prescription_timestamp").isNotNull()
                & (
                    to_date(col("prescription_timestamp"))
                    < lit("2000-01-01").cast("date")
                ),
                lit("Prescription timestamp before allowed date")
            ),
            when(
                col("prescription_timestamp").isNotNull()
                & (
                    to_date(col("prescription_timestamp"))
                    > current_date()
                ),
                lit("Prescription timestamp is in the future")
            ),

            # Drug code validation
            when(
                col("drug_ndc_code").isNull(),
                lit("Missing drug_ndc_code")
            ),
            when(
                col("drug_ndc_code").isNotNull()
                & ~col("drug_ndc_code")
                    .rlike(r"^[0-9]{5}-[0-9]{4}-[0-9]{2}$"),
                lit("Invalid drug_ndc_code format")
            ),

            # Drug-name validation
            when(
                col("drug_name").isNull(),
                lit("Missing drug_name")
            ),
            when(
                col("drug_name").isNotNull()
                & lower(col("drug_name")).contains("<script"),
                lit("Invalid script content in drug_name")
            ),

            # Dosage validation
            when(
                col("dosage").isNull(),
                lit("Missing dosage")
            ),
            when(
                col("dosage").isNotNull()
                & lower(col("dosage")).contains("<script"),
                lit("Invalid script content in dosage")
            ),
            when(
                col("dosage").isNotNull()
                & col("dosage").contains("#"),
                lit("Invalid dosage format")
            ),

            # Quantity validation
            when(
                col("quantity_raw").isNull(),
                lit("Missing quantity")
            ),
            when(
                col("quantity_raw").isNotNull()
                & col("quantity").isNull(),
                lit("Quantity is not a valid whole number")
            ),
            when(
                col("quantity").isNotNull()
                & (
                    (col("quantity") <= 0)
                    | (col("quantity") > 10000)
                ),
                lit("Quantity outside allowed range")
            ),

            # Refill validation
            when(
                col("refills_authorized_raw").isNull(),
                lit("Missing refills_authorized")
            ),
            when(
                col("refills_authorized_raw").isNotNull()
                & col("refills_authorized").isNull(),
                lit("Refills authorized is not a valid whole number")
            ),
            when(
                col("refills_authorized").isNotNull()
                & (
                    (col("refills_authorized") < 0)
                    | (col("refills_authorized") > 99)
                ),
                lit("Refills authorized outside allowed range")
            ),

            # Status validation
            when(
                col("dispense_status").isNull(),
                lit("Missing dispense_status")
            ),
            when(
                col("dispense_status").isNotNull()
                & ~col("dispense_status").isin(
                    allowed_dispense_statuses
                ),
                lit("Invalid dispense_status")
            ),

            # Denial reason validation
            when(
                col("denial_reason").isNotNull()
                & lower(col("denial_reason")).contains("<script"),
                lit("Invalid script content in denial_reason")
            ),

            # IP validation when present
            when(
                col("pharmacy_ip_address").isNotNull()
                & ~col("pharmacy_ip_address").rlike(
                    r"^(25[0-5]|2[0-4][0-9]|1?[0-9]{1,2})"
                    r"\.(25[0-5]|2[0-4][0-9]|1?[0-9]{1,2})"
                    r"\.(25[0-5]|2[0-4][0-9]|1?[0-9]{1,2})"
                    r"\.(25[0-5]|2[0-4][0-9]|1?[0-9]{1,2})$"
                ),
                lit("Invalid pharmacy_ip_address")
            ),

            # Unexpected source field should normally be empty
            when(
                col("unexpected_source_field").isNotNull(),
                lit("Unexpected source field contains a value")
            ),

            # Required technical metadata
            when(
                col("batch_id").isNull(),
                lit("Missing batch_id")
            ),
            when(
                col("source_file_name").isNull(),
                lit("Missing source_file_name")
            ),
            when(
                col("ingestion_timestamp").isNull(),
                lit("Missing ingestion_timestamp")
            )
        )
    )
)

# Valid records: remove validation helper columns before Silver
valid_prescriptions_df = (
    prescriptions_validated_df
    .filter(col("error_reason") == "")
    .drop(
        "error_reason",
        "prescription_timestamp_raw",
        "quantity_raw",
        "refills_authorized_raw"
    )
)

# Rejected records retain raw values for investigation
quarantine_prescriptions_df = (
    prescriptions_validated_df
    .filter(col("error_reason") != "")
)

print("Total Rows:", prescriptions_validated_df.count())
print("Valid Rows:", valid_prescriptions_df.count())
print("Rejected Rows:", quarantine_prescriptions_df.count())

display(
    quarantine_prescriptions_df.select(
        "prescription_id",
        "patient_id",
        "prescribing_provider_id",
        "pharmacy_id",
        "prescription_timestamp_raw",
        "drug_ndc_code",
        "drug_name",
        "dosage",
        "quantity_raw",
        "refills_authorized_raw",
        "dispense_status",
        "pharmacy_ip_address",
        "unexpected_source_field",
        "error_reason"
    ).limit(20)
)

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 29, Finished, Available, Finished, False)

Total Rows: 486634
Valid Rows: 1618
Rejected Rows: 485016


SynapseWidget(Synapse.DataFrame, de74bc56-677a-4208-82d1-fbcf7a2d0b41)

In [28]:
# CELL 5 - HANDLE DUPLICATE PRESCRIPTION RECORDS
# Deterministic deduplication: one record per prescription_id

from pyspark.sql import Window
from pyspark.sql.functions import (
    col,
    row_number,
    lit,
    sha2,
    concat_ws,
    coalesce
)

# Columns used to create a stable tie-breaker
dedup_hash_columns = [
    "prescription_id",
    "patient_id",
    "prescribing_provider_id",
    "pharmacy_id",
    "prescription_timestamp",
    "drug_ndc_code",
    "drug_name",
    "dosage",
    "quantity",
    "refills_authorized",
    "dispense_status",
    "denial_reason",
    "pharmacy_ip_address",
    "unexpected_source_field",
    "batch_id",
    "source_file_name",
    "ingestion_timestamp"
]

# Create a deterministic hash for identical timestamp ties
valid_prescriptions_with_hash_df = (
    valid_prescriptions_df
    .withColumn(
        "_dedup_row_hash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("<NULL>")
                    )
                    for column_name in dedup_hash_columns
                ]
            ),
            256
        )
    )
)

# Keep the most recently ingested record.
# If ingestion timestamps match, use prescription timestamp.
# If both match, use the stable row hash.
duplicate_window = (
    Window
    .partitionBy("prescription_id")
    .orderBy(
        col("ingestion_timestamp").desc_nulls_last(),
        col("prescription_timestamp").desc_nulls_last(),
        col("_dedup_row_hash").desc()
    )
)

ranked_prescriptions_df = (
    valid_prescriptions_with_hash_df
    .withColumn(
        "record_rank",
        row_number().over(duplicate_window)
    )
)

# Keep exactly one deterministic record per prescription_id
deduplicated_prescriptions_df = (
    ranked_prescriptions_df
    .filter(col("record_rank") == 1)
    .drop(
        "record_rank",
        "_dedup_row_hash"
    )
)

# Send additional valid duplicate records to quarantine
duplicate_prescriptions_df = (
    ranked_prescriptions_df
    .filter(col("record_rank") > 1)
    .drop(
        "record_rank",
        "_dedup_row_hash"
    )
    .withColumn(
        "error_reason",
        lit("Duplicate prescription_id")
    )
)

# Combine validation failures and duplicate failures
final_quarantine_prescriptions_df = (
    quarantine_prescriptions_df
    .unionByName(
        duplicate_prescriptions_df,
        allowMissingColumns=True
    )
)

valid_before_dedup_count = valid_prescriptions_df.count()
duplicate_count = duplicate_prescriptions_df.count()
silver_candidate_count = deduplicated_prescriptions_df.count()
final_quarantine_count = final_quarantine_prescriptions_df.count()

print(
    "Valid rows before deduplication:",
    valid_before_dedup_count
)

print(
    "Duplicate rows sent to quarantine:",
    duplicate_count
)

print(
    "Final Silver candidate rows:",
    silver_candidate_count
)

print(
    "Total quarantine rows:",
    final_quarantine_count
)

# Confirm valid rows are fully accounted for
if valid_before_dedup_count != (
    silver_candidate_count + duplicate_count
):
    raise ValueError(
        "Deduplication reconciliation failed. "
        f"Valid before dedup={valid_before_dedup_count}, "
        f"Silver candidates + duplicates="
        f"{silver_candidate_count + duplicate_count}"
    )

print("SUCCESS: Deduplication reconciliation completed.")

display(
    duplicate_prescriptions_df
    .select(
        "prescription_id",
        "patient_id",
        "prescription_timestamp",
        "drug_name",
        "quantity",
        "refills_authorized",
        "dispense_status",
        "batch_id",
        "ingestion_timestamp",
        "error_reason"
    )
    .limit(20)
)

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 30, Finished, Available, Finished, False)

Valid rows before deduplication: 1618
Duplicate rows sent to quarantine: 67
Final Silver candidate rows: 1551
Total quarantine rows: 485083
SUCCESS: Deduplication reconciliation completed.


SynapseWidget(Synapse.DataFrame, c6a3c98a-c3da-44ae-bc75-97e89912b423)

In [29]:
# CELL 6 - IDEMPOTENT UPSERT INTO SILVER PRESCRIPTIONS TABLE

from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    current_timestamp,
    sha2,
    concat_ws,
    coalesce,
    lit
)

silver_table_name = "silver_prescriptions"

# Business columns used to detect genuine changes
hash_columns = [
    "prescription_id",
    "patient_id",
    "prescribing_provider_id",
    "pharmacy_id",
    "prescription_timestamp",
    "drug_ndc_code",
    "drug_name",
    "dosage",
    "quantity",
    "refills_authorized",
    "dispense_status",
    "denial_reason",
    "pharmacy_ip_address",
    "unexpected_source_field"
]

# Confirm required columns exist
missing_hash_columns = [
    column_name
    for column_name in hash_columns
    if column_name not in deduplicated_prescriptions_df.columns
]

if missing_hash_columns:
    raise ValueError(
        f"Missing required Silver columns: {missing_hash_columns}"
    )

# Cell 5 must already guarantee one row per prescription_id
remaining_duplicate_count = (
    deduplicated_prescriptions_df
    .groupBy("prescription_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

if remaining_duplicate_count > 0:
    raise ValueError(
        "deduplicated_prescriptions_df still contains duplicate "
        "prescription_id values. Recheck Cell 5."
    )

silver_source_df = (
    deduplicated_prescriptions_df
    .withColumn(
        "row_hash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("<NULL>")
                    )
                    for column_name in hash_columns
                ]
            ),
            256
        )
    )
    .withColumn(
        "silver_processing_timestamp",
        current_timestamp()
    )
)

source_count = silver_source_df.count()

print("Silver source rows:", source_count)

if not spark.catalog.tableExists(silver_table_name):

    (
        silver_source_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table_name)
    )

    print("Silver prescriptions table created.")

else:

    target_df = spark.table(silver_table_name)
    existing_columns = target_df.columns

    # Add metadata columns if the table came from an older notebook
    if "row_hash" not in existing_columns:
        spark.sql(f"""
            ALTER TABLE {silver_table_name}
            ADD COLUMNS (row_hash STRING)
        """)

        print("row_hash column added.")

    if "silver_processing_timestamp" not in existing_columns:
        spark.sql(f"""
            ALTER TABLE {silver_table_name}
            ADD COLUMNS (
                silver_processing_timestamp TIMESTAMP
            )
        """)

        print("silver_processing_timestamp column added.")

    # Reload schema after any ALTER TABLE operations
    target_columns = spark.table(silver_table_name).columns
    source_columns = silver_source_df.columns

    missing_target_columns = [
        column_name
        for column_name in source_columns
        if column_name not in target_columns
    ]

    if missing_target_columns:
        raise ValueError(
            "Silver target table is missing source columns: "
            f"{missing_target_columns}. "
            "Update the table schema before running the merge."
        )

    silver_delta_table = DeltaTable.forName(
        spark,
        silver_table_name
    )

    (
        silver_delta_table.alias("target")
        .merge(
            silver_source_df.alias("source"),
            "target.prescription_id = source.prescription_id"
        )
        .whenMatchedUpdateAll(
            condition="""
                target.row_hash IS NULL
                OR target.row_hash <> source.row_hash
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Silver prescriptions table merged successfully.")

silver_row_count = (
    spark.table(silver_table_name)
    .count()
)

print("Silver table:", silver_table_name)
print("Final Silver table rows:", silver_row_count)

display(
    spark.table(silver_table_name)
    .orderBy("prescription_id")
    .limit(20)
)

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 31, Finished, Available, Finished, False)

Silver source rows: 1551
Silver prescriptions table merged successfully.
Silver table: silver_prescriptions
Final Silver table rows: 92987


SynapseWidget(Synapse.DataFrame, 03b5dc7c-4f12-4c77-b1f8-a77ab3b6765b)

In [30]:
# CELL 7 - IDEMPOTENT UPSERT INTO QUARANTINE PRESCRIPTIONS TABLE
# Adds missing columns and aligns source datatypes with the target table

from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    current_timestamp,
    sha2,
    concat_ws,
    coalesce,
    lit
)

quarantine_table_name = "quarantine_prescriptions"

quarantine_hash_columns = sorted(
    [
        column_name
        for column_name in final_quarantine_prescriptions_df.columns
        if column_name not in [
            "quarantine_record_id",
            "quarantine_timestamp"
        ]
    ]
)

if "error_reason" not in quarantine_hash_columns:
    raise ValueError(
        "final_quarantine_prescriptions_df is missing error_reason."
    )

quarantine_source_df = (
    final_quarantine_prescriptions_df
    .withColumn(
        "quarantine_record_id",
        sha2(
            concat_ws(
                "||",
                *[
                    concat_ws(
                        "=",
                        lit(column_name),
                        coalesce(
                            col(column_name).cast("string"),
                            lit("<NULL>")
                        )
                    )
                    for column_name in quarantine_hash_columns
                ]
            ),
            256
        )
    )
    .dropDuplicates(["quarantine_record_id"])
    .withColumn(
        "quarantine_timestamp",
        current_timestamp()
    )
)

print(
    "Quarantine rows before exact deduplication:",
    final_quarantine_prescriptions_df.count()
)

print(
    "Quarantine rows after exact deduplication:",
    quarantine_source_df.count()
)

duplicate_key_count = (
    quarantine_source_df
    .groupBy("quarantine_record_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_key_count > 0:
    raise ValueError(
        "Duplicate quarantine_record_id values detected."
    )

if not spark.catalog.tableExists(quarantine_table_name):

    (
        quarantine_source_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(quarantine_table_name)
    )

    print("Quarantine prescriptions table created.")

else:

    target_schema = spark.table(
        quarantine_table_name
    ).schema

    target_types = {
        field.name: field.dataType
        for field in target_schema.fields
    }

    if "quarantine_record_id" not in target_types:
        raise ValueError(
            f"{quarantine_table_name} exists without "
            "quarantine_record_id."
        )

    sql_type_map = {
        "string": "STRING",
        "int": "INT",
        "bigint": "BIGINT",
        "long": "BIGINT",
        "timestamp": "TIMESTAMP",
        "date": "DATE",
        "double": "DOUBLE",
        "float": "FLOAT",
        "boolean": "BOOLEAN"
    }

    missing_source_fields = [
        field
        for field in quarantine_source_df.schema.fields
        if field.name not in target_types
    ]

    for field in missing_source_fields:

        spark_type = field.dataType.simpleString()

        sql_type = sql_type_map.get(
            spark_type,
            spark_type.upper()
        )

        spark.sql(f"""
            ALTER TABLE {quarantine_table_name}
            ADD COLUMNS (
                `{field.name}` {sql_type}
            )
        """)

        print(
            f"Added missing quarantine column: "
            f"{field.name} {sql_type}"
        )

    updated_target_schema = spark.table(
        quarantine_table_name
    ).schema

    updated_target_types = {
        field.name: field.dataType
        for field in updated_target_schema.fields
    }

    # Cast source columns to existing target datatypes
    for source_field in quarantine_source_df.schema.fields:

        column_name = source_field.name

        if column_name in updated_target_types:

            target_type = updated_target_types[column_name]

            if source_field.dataType != target_type:

                print(
                    f"Casting {column_name}: "
                    f"{source_field.dataType.simpleString()} -> "
                    f"{target_type.simpleString()}"
                )

                quarantine_source_df = (
                    quarantine_source_df
                    .withColumn(
                        column_name,
                        col(column_name).cast(target_type)
                    )
                )

    quarantine_delta_table = DeltaTable.forName(
        spark,
        quarantine_table_name
    )

    (
        quarantine_delta_table.alias("target")
        .merge(
            quarantine_source_df.alias("source"),
            """
            target.quarantine_record_id =
            source.quarantine_record_id
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        "Quarantine prescriptions table merged successfully."
    )

print(
    "Final quarantine table rows:",
    spark.table(quarantine_table_name).count()
)

display(
    spark.table(quarantine_table_name)
    .select(
        "quarantine_record_id",
        "prescription_id",
        "patient_id",
        "batch_id",
        "source_file_name",
        "prescription_timestamp_raw",
        "quantity_raw",
        "refills_authorized_raw",
        "error_reason",
        "quarantine_timestamp"
    )
    .orderBy(
        col("prescription_id").asc_nulls_last()
    )
    .limit(20)
)

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 32, Finished, Available, Finished, False)

Quarantine rows before exact deduplication: 485083
Quarantine rows after exact deduplication: 466031
Casting quantity: bigint -> int
Quarantine prescriptions table merged successfully.
Final quarantine table rows: 842410


SynapseWidget(Synapse.DataFrame, 93d81562-7fe1-4b53-941a-2f8c527bc589)

In [31]:
# CELL 8 - UPSERT PRESCRIPTIONS AUDIT RECORD
# Rerunnable: one audit row per batch_id + dataset_name

from pyspark.sql.functions import (
    col,
    xxhash64,
    concat_ws,
    coalesce,
    lit
)

audit_table = "healthcare_control_lakehouse.dbo.audit_table"
dataset_name = "prescriptions"

# Select the batch being processed.
# For the current notebook run, Bronze must contain one batch.
batch_ids = [
    row["batch_id"]
    for row in (
        bronze_prescriptions_df
        .select("batch_id")
        .where(col("batch_id").isNotNull())
        .distinct()
        .collect()
    )
]

if len(batch_ids) != 1:
    raise ValueError(
        "This notebook run must process exactly one batch_id. "
        f"Found: {batch_ids}"
    )

current_batch_id = batch_ids[0]

# Current-batch counts
rows_received = (
    bronze_prescriptions_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

rows_written = (
    deduplicated_prescriptions_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

rejected_rows = (
    final_quarantine_prescriptions_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

# Reconcile before recording success
if rows_received != rows_written + rejected_rows:
    raise ValueError(
        f"Audit reconciliation failed for batch "
        f"{current_batch_id}. "
        f"Received={rows_received}, "
        f"Written+Rejected={rows_written + rejected_rows}"
    )

print("Batch ID:", current_batch_id)
print("Dataset:", dataset_name)
print("Rows received:", rows_received)
print("Rows written:", rows_written)
print("Rejected rows:", rejected_rows)

# Confirm audit table exists
if not spark.catalog.tableExists(audit_table):
    raise ValueError(
        f"Audit table does not exist: {audit_table}"
    )

audit_schema = spark.table(audit_table).schema
audit_columns = spark.table(audit_table).columns

# Build source record
audit_source_df = spark.createDataFrame(
    [
        (
            current_batch_id,
            dataset_name,
            rows_received,
            rows_written,
            rejected_rows,
            "COMPLETED"
        )
    ],
    """
    batch_id string,
    dataset_name string,
    rows_received long,
    rows_written long,
    rejected_rows long,
    status string
    """
)

# Add deterministic audit_id only when required
if "audit_id" in audit_columns:

    audit_id_type = next(
        field.dataType.simpleString()
        for field in audit_schema.fields
        if field.name == "audit_id"
    )

    if audit_id_type not in ["bigint", "long"]:
        raise ValueError(
            "audit_id exists but is not LONG/BIGINT. "
            f"Current datatype: {audit_id_type}"
        )

    audit_source_df = (
        audit_source_df
        .withColumn(
            "audit_id",
            xxhash64(
                concat_ws(
                    "||",
                    coalesce(col("batch_id"), lit("<NULL>")),
                    coalesce(col("dataset_name"), lit("<NULL>"))
                )
            )
        )
    )

audit_source_df.createOrReplaceTempView(
    "prescriptions_audit_source"
)

# Merge using only columns supported by the target table
if "audit_id" in audit_columns:

    spark.sql(f"""
        MERGE INTO {audit_table} AS target

        USING (
            SELECT
                audit_id,
                batch_id,
                dataset_name,
                current_timestamp() AS start_timestamp,
                current_timestamp() AS end_timestamp,
                rows_received,
                rows_written,
                rejected_rows,
                status
            FROM prescriptions_audit_source
        ) AS source

        ON target.batch_id = source.batch_id
        AND target.dataset_name = source.dataset_name

        WHEN MATCHED THEN UPDATE SET
            target.end_timestamp = source.end_timestamp,
            target.rows_received = source.rows_received,
            target.rows_written = source.rows_written,
            target.rejected_rows = source.rejected_rows,
            target.status = source.status

        WHEN NOT MATCHED THEN INSERT (
            audit_id,
            batch_id,
            dataset_name,
            start_timestamp,
            end_timestamp,
            rows_received,
            rows_written,
            rejected_rows,
            status
        )
        VALUES (
            source.audit_id,
            source.batch_id,
            source.dataset_name,
            source.start_timestamp,
            source.end_timestamp,
            source.rows_received,
            source.rows_written,
            source.rejected_rows,
            source.status
        )
    """)

else:

    spark.sql(f"""
        MERGE INTO {audit_table} AS target

        USING (
            SELECT
                batch_id,
                dataset_name,
                current_timestamp() AS start_timestamp,
                current_timestamp() AS end_timestamp,
                rows_received,
                rows_written,
                rejected_rows,
                status
            FROM prescriptions_audit_source
        ) AS source

        ON target.batch_id = source.batch_id
        AND target.dataset_name = source.dataset_name

        WHEN MATCHED THEN UPDATE SET
            target.end_timestamp = source.end_timestamp,
            target.rows_received = source.rows_received,
            target.rows_written = source.rows_written,
            target.rejected_rows = source.rejected_rows,
            target.status = source.status

        WHEN NOT MATCHED THEN INSERT (
            batch_id,
            dataset_name,
            start_timestamp,
            end_timestamp,
            rows_received,
            rows_written,
            rejected_rows,
            status
        )
        VALUES (
            source.batch_id,
            source.dataset_name,
            source.start_timestamp,
            source.end_timestamp,
            source.rows_received,
            source.rows_written,
            source.rejected_rows,
            source.status
        )
    """)

print(
    "Prescriptions audit record inserted or updated successfully."
)

display(
    spark.table(audit_table)
    .filter(
        (col("batch_id") == current_batch_id)
        & (col("dataset_name") == dataset_name)
    )
)

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 33, Finished, Available, Finished, False)

Batch ID: prescriptions_batch_001
Dataset: prescriptions
Rows received: 486634
Rows written: 1551
Rejected rows: 485083
Prescriptions audit record inserted or updated successfully.


SynapseWidget(Synapse.DataFrame, d30c7631-7dae-4faf-8072-7237585a514f)

In [32]:
# CELL 9 - INSPECT FILE TRACKING TABLE

file_tracking_table = (
    "healthcare_control_lakehouse.dbo.file_tracking_table"
)

file_tracking_df = spark.table(file_tracking_table)

print("File-tracking table schema:")
file_tracking_df.printSchema()

print("Columns:")
print(file_tracking_df.columns)

display(file_tracking_df.limit(20))

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 34, Finished, Available, Finished, False)

File-tracking table schema:
root
 |-- file_tracking_id: long (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- dataset_name: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)
 |-- processing_status: string (nullable = true)

Columns:
['file_tracking_id', 'source_file_name', 'dataset_name', 'batch_id', 'processed_timestamp', 'processing_status']


SynapseWidget(Synapse.DataFrame, 9fc3be85-ae6b-4891-8565-b72e81b01ef3)

In [33]:
# CELL 10 - UPSERT PRESCRIPTIONS FILE-TRACKING RECORD
# Rerunnable and compatible with existing notebook cells

from pyspark.sql.functions import (
    col,
    concat_ws,
    coalesce,
    lit,
    xxhash64
)

file_tracking_table = (
    "healthcare_control_lakehouse.dbo.file_tracking_table"
)

dataset_name = "prescriptions"

# Reuse the batch selected in Cell 8
if "current_batch_id" not in globals():
    raise ValueError(
        "current_batch_id is not available. "
        "Run Cell 8 before Cell 10."
    )

# Confirm the target table exists
if not spark.catalog.tableExists(file_tracking_table):
    raise ValueError(
        f"File-tracking table does not exist: "
        f"{file_tracking_table}"
    )

# Get every source file belonging to the current batch
current_batch_files_df = (
    bronze_prescriptions_df
    .filter(col("batch_id") == current_batch_id)
    .select("source_file_name")
    .where(col("source_file_name").isNotNull())
    .distinct()
)

source_file_count = current_batch_files_df.count()

if source_file_count == 0:
    raise ValueError(
        f"No source files found for batch_id="
        f"{current_batch_id}"
    )

# Create one tracking row per source file
file_tracking_source_df = (
    current_batch_files_df
    .withColumn(
        "dataset_name",
        lit(dataset_name)
    )
    .withColumn(
        "batch_id",
        lit(current_batch_id)
    )
    .withColumn(
        "processing_status",
        lit("COMPLETED")
    )
    .withColumn(
        "file_tracking_id",
        xxhash64(
            concat_ws(
                "||",
                coalesce(
                    col("source_file_name"),
                    lit("<NULL>")
                ),
                coalesce(
                    col("dataset_name"),
                    lit("<NULL>")
                ),
                coalesce(
                    col("batch_id"),
                    lit("<NULL>")
                )
            )
        )
    )
    .select(
        "file_tracking_id",
        "source_file_name",
        "dataset_name",
        "batch_id",
        "processing_status"
    )
)

# Confirm one unique source row per file
duplicate_source_count = (
    file_tracking_source_df
    .groupBy(
        "source_file_name",
        "dataset_name",
        "batch_id"
    )
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_source_count > 0:
    raise ValueError(
        "Duplicate file-tracking source rows detected."
    )

# Verify target schema
target_schema = spark.table(file_tracking_table).schema
target_columns = spark.table(file_tracking_table).columns

required_target_columns = [
    "file_tracking_id",
    "source_file_name",
    "dataset_name",
    "batch_id",
    "processed_timestamp",
    "processing_status"
]

missing_target_columns = [
    column_name
    for column_name in required_target_columns
    if column_name not in target_columns
]

if missing_target_columns:
    raise ValueError(
        "File-tracking table is missing required columns: "
        f"{missing_target_columns}"
    )

file_tracking_id_type = next(
    field.dataType.simpleString()
    for field in target_schema.fields
    if field.name == "file_tracking_id"
)

if file_tracking_id_type not in ["bigint", "long"]:
    raise ValueError(
        "file_tracking_id must be LONG/BIGINT. "
        f"Current datatype: {file_tracking_id_type}"
    )

file_tracking_source_df.createOrReplaceTempView(
    "prescriptions_file_tracking_source"
)

spark.sql(f"""
MERGE INTO {file_tracking_table} AS target

USING (
    SELECT
        file_tracking_id,
        source_file_name,
        dataset_name,
        batch_id,
        current_timestamp() AS processed_timestamp,
        processing_status
    FROM prescriptions_file_tracking_source
) AS source

ON target.source_file_name = source.source_file_name
AND target.dataset_name = source.dataset_name
AND target.batch_id = source.batch_id

WHEN MATCHED THEN UPDATE SET
    target.processing_status = source.processing_status

WHEN NOT MATCHED THEN INSERT (
    file_tracking_id,
    source_file_name,
    dataset_name,
    batch_id,
    processed_timestamp,
    processing_status
)
VALUES (
    source.file_tracking_id,
    source.source_file_name,
    source.dataset_name,
    source.batch_id,
    source.processed_timestamp,
    source.processing_status
)
""")

print(
    "Prescriptions file-tracking records "
    "inserted or updated successfully."
)

print("Batch ID:", current_batch_id)
print("Dataset:", dataset_name)
print("Source file count:", source_file_count)

display(
    spark.table(file_tracking_table)
    .filter(
        (col("dataset_name") == dataset_name)
        & (col("batch_id") == current_batch_id)
    )
    .orderBy("source_file_name")
)

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 35, Finished, Available, Finished, False)

Prescriptions file-tracking records inserted or updated successfully.
Batch ID: prescriptions_batch_001
Dataset: prescriptions
Source file count: 1


SynapseWidget(Synapse.DataFrame, ca60abff-2d94-4dc9-9b8e-f77f421b9114)

In [34]:
# CELL 11 - FINAL RECONCILIATION CHECK
# Validates source classification and confirms current records reached targets

from pyspark.sql.functions import col

if "current_batch_id" not in globals():
    raise ValueError(
        "current_batch_id is unavailable. Run Cell 8 first."
    )

# ---------------------------------------------------------
# 1. CURRENT-BATCH SOURCE RECONCILIATION
# ---------------------------------------------------------

bronze_count = (
    bronze_prescriptions_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

silver_candidate_count = (
    deduplicated_prescriptions_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

quarantine_source_count = (
    final_quarantine_prescriptions_df
    .filter(col("batch_id") == current_batch_id)
    .count()
)

reconciled_count = (
    silver_candidate_count
    + quarantine_source_count
)

print("Batch ID:", current_batch_id)
print("Bronze rows:", bronze_count)
print("Silver candidate rows:", silver_candidate_count)
print("Quarantine source rows:", quarantine_source_count)
print("Reconciled rows:", reconciled_count)

if bronze_count != reconciled_count:
    raise ValueError(
        f"Source reconciliation failed for batch "
        f"{current_batch_id}. "
        f"Bronze={bronze_count}, "
        f"Silver candidates + quarantine={reconciled_count}"
    )

print(
    "SUCCESS: Every Bronze row is classified as "
    "Silver or Quarantine."
)

# ---------------------------------------------------------
# 2. CONFIRM CURRENT SILVER CANDIDATES EXIST IN TARGET
# ---------------------------------------------------------

if not spark.catalog.tableExists("silver_prescriptions"):
    raise ValueError(
        "Silver table does not exist: silver_prescriptions"
    )

current_silver_source_df = (
    silver_source_df
    .filter(col("batch_id") == current_batch_id)
    .select(
        "prescription_id",
        "row_hash"
    )
    .dropDuplicates(["prescription_id"])
)

silver_target_check_df = (
    spark.table("silver_prescriptions")
    .select(
        "prescription_id",
        "row_hash"
    )
)

missing_or_mismatched_silver_count = (
    current_silver_source_df.alias("source")
    .join(
        silver_target_check_df.alias("target"),
        (
            col("source.prescription_id")
            == col("target.prescription_id")
        )
        & (
            col("source.row_hash")
            == col("target.row_hash")
        ),
        "left_anti"
    )
    .count()
)

if missing_or_mismatched_silver_count > 0:
    raise ValueError(
        f"{missing_or_mismatched_silver_count} current Silver "
        "candidate records are missing or do not match the "
        "target table."
    )

print(
    "SUCCESS: All current Silver candidates exist in "
    "silver_prescriptions with matching row_hash."
)

# ---------------------------------------------------------
# 3. CONFIRM CURRENT QUARANTINE RECORDS EXIST IN TARGET
# ---------------------------------------------------------

if not spark.catalog.tableExists("quarantine_prescriptions"):
    raise ValueError(
        "Quarantine table does not exist: "
        "quarantine_prescriptions"
    )

current_quarantine_source_df = (
    quarantine_source_df
    .filter(col("batch_id") == current_batch_id)
    .select("quarantine_record_id")
    .dropDuplicates(["quarantine_record_id"])
)

quarantine_target_check_df = (
    spark.table("quarantine_prescriptions")
    .select("quarantine_record_id")
)

missing_quarantine_count = (
    current_quarantine_source_df.alias("source")
    .join(
        quarantine_target_check_df.alias("target"),
        col("source.quarantine_record_id")
        == col("target.quarantine_record_id"),
        "left_anti"
    )
    .count()
)

if missing_quarantine_count > 0:
    raise ValueError(
        f"{missing_quarantine_count} current quarantine "
        "records are missing from the target table."
    )

print(
    "SUCCESS: All current quarantine records exist in "
    "quarantine_prescriptions."
)

# ---------------------------------------------------------
# 4. CHECK AUDIT RECORD
# ---------------------------------------------------------

audit_table = (
    "healthcare_control_lakehouse.dbo.audit_table"
)

audit_record_count = (
    spark.table(audit_table)
    .filter(
        (col("batch_id") == current_batch_id)
        & (col("dataset_name") == "prescriptions")
        & (col("status") == "COMPLETED")
    )
    .count()
)

if audit_record_count != 1:
    raise ValueError(
        f"Expected exactly one completed audit record for "
        f"{current_batch_id}, but found {audit_record_count}."
    )

print("Completed audit records:", audit_record_count)

# ---------------------------------------------------------
# 5. CHECK FILE-TRACKING RECORDS
# ---------------------------------------------------------

file_tracking_table = (
    "healthcare_control_lakehouse.dbo.file_tracking_table"
)

expected_file_count = (
    bronze_prescriptions_df
    .filter(col("batch_id") == current_batch_id)
    .select("source_file_name")
    .where(col("source_file_name").isNotNull())
    .distinct()
    .count()
)

completed_file_count = (
    spark.table(file_tracking_table)
    .filter(
        (col("batch_id") == current_batch_id)
        & (col("dataset_name") == "prescriptions")
        & (col("processing_status") == "COMPLETED")
    )
    .select("source_file_name")
    .distinct()
    .count()
)

if expected_file_count != completed_file_count:
    raise ValueError(
        f"File-tracking reconciliation failed. "
        f"Expected files={expected_file_count}, "
        f"completed files={completed_file_count}"
    )

print("Expected source files:", expected_file_count)
print("Completed tracked files:", completed_file_count)

print(
    "SUCCESS: Final prescriptions pipeline "
    "reconciliation completed."
)

StatementMeta(, b683fbeb-1f7a-44a1-876e-78262fd49149, 36, Finished, Available, Finished, False)

Batch ID: prescriptions_batch_001
Bronze rows: 486634
Silver candidate rows: 1551
Quarantine source rows: 485083
Reconciled rows: 486634
SUCCESS: Every Bronze row is classified as Silver or Quarantine.
SUCCESS: All current Silver candidates exist in silver_prescriptions with matching row_hash.
SUCCESS: All current quarantine records exist in quarantine_prescriptions.
Completed audit records: 1
Expected source files: 1
Completed tracked files: 1
SUCCESS: Final prescriptions pipeline reconciliation completed.
